                                                     REAL TIME OBJECT DETECTION SYSTEM

In [2]:
from ultralytics import YOLO
import supervision as sv
import cv2
import time 
import csv
from datetime import datetime 
from collections import Counter 

model = YOLO("yolov8n.pt")

In [9]:
tracker = sv.ByteTrack()
box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()


#Line Zoning
LINE_START = sv.Point(0,240)
LINE_END = sv.Point(640,240)
Line_Counter = sv.LineZone(start = LINE_START , end = LINE_END)
Line_Annotator = sv.LineZoneAnnotator(thickness = 1)

#CSV Logging setup
csv_file = open("Detection_log.csv","w",newline ="")
csv_writer = csv.writer(csv_file)
csv_writer.writerow(["timestamp","tracker_id","class","confidence","x1","y1","x2","y2"])

#FPS Tracking
prev_time = time.time()
fps = 0

#Class Counter
class_totals = Counter()

#Video Output

out = cv2.VideoWriter("output.mp4",cv2.VideoWriter_fourcc(*"mp4v"),30,(640,480))
print("Running - press c to exit")

seen_ids = set()


cap = cv2.VideoCapture(0)

#Main Loop
while True:
    ret , frame = cap.read()
    if not ret:
        break

    curr_time = time.time()
    fps = 1 / (curr_time - prev_time)
    prev_time = curr_time
        
    results = model(frame , verbose = False , conf = 0.4)[0]
    detections = sv.Detections.from_ultralytics(results)
    detections = tracker.update_with_detections(detections)

    crossed_in, crossed_out = Line_Counter.trigger(detections)

    Labels = [f"#{tracker_id} {model.names[class_id]} {conf:.2f}"
              for class_id , conf , tracker_id
              in zip (detections.class_id , detections.confidence , detections.tracker_id)
             ]

    #CSV LOGGING

    timestamp = datetime.now().strftime("%H:%M:%S.%f")[:-3]
    for i , (class_id , conf , tracker_id , box) in enumerate(zip(
        detections.class_id,
        detections.confidence,
        detections.tracker_id,
        detections.xyxy
    )):
        x1,y1,x2,y2 = map(int , box)
        class_name = model.names[class_id]
        csv_writer.writerow([timestamp,tracker_id,class_name,f"{conf:.2f}" , x1 , y1, x2, y2])
        if tracker_id not in seen_ids:
            seen_ids.add(tracker_id)
            class_totals[class_name] += 1 

    #Drawing Boxes , labels , lines 
    
    frame = box_annotator.annotate(frame , detections)
    frame = label_annotator.annotate(frame , detections , Labels)
    frame = Line_Annotator.annotate(frame , Line_Counter)

    #HUD - Top left

    cv2.putText(frame, f"FPS: {fps:.1f}",
                (20, 40),  cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2)
    cv2.putText(frame, f"IN:  {Line_Counter.in_count}",
                (20, 75),  cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0),   2)
    cv2.putText(frame, f"OUT: {Line_Counter.out_count}",
                (20, 110), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255),   2)

    
    # Class Totals Sidebar
    
    y_offset = 40
    cv2.putText(frame , "Detected:" , (440, y_offset),
                cv2.FONT_HERSHEY_SIMPLEX , 0.6 , (255,0,0),1)
    for class_name , count in class_totals.most_common(5):
        y_offset += 28
        cv2.putText(frame , f"{class_name} : {count}",
                    (440 , y_offset),cv2.FONT_HERSHEY_SIMPLEX , 0.6 , (0,0,255),1)

    out.write(frame)
        
                

    cv2.imshow("Tracker", frame)
    if cv2.waitKey(1) & 0xFF == ord('c'):
        break


cap.release()
out.release()
csv_file.close()
cv2.waitKey(1)
cv2.destroyAllWindows()
cv2.waitKey(1)

print(f"\nDone. Final counts — IN: {Line_Counter.in_count}  OUT: {Line_Counter.out_count}")
print("Saved: output.mp4 and detections_log.csv")

Running - press c to exit

Done. Final counts — IN: 0  OUT: 0
Saved: output.mp4 and detections_log.csv
